# 实验 1：$G$ 的峰能否稳定定位 Hopfield 吸引域边界？

实验 0 只展示了一个漂亮现象。这里把它变成可否证的定量检查：

1. 用长时终点的 A/B overlap margin 二分得到独立边界 $\kappa^*$；
2. 在每个时间切片上估计 $G(t,\kappa)$ 的峰 $\hat\kappa_G(t)$；
3. 计算定位误差 $e(t)=|\hat\kappa_G(t)-\kappa^*|$；
4. 扫描 8 个 seed 和 4 个 A/B overlap。

模型仍是实验 0 的连续经典 Hopfield：

$$\dot x=-x+W\tanh(gx),\qquad x_0(\kappa)=(1-\kappa)A+\kappa B.$$

这是**方法校准**，不是论文复现，也不比较 AUC 或其他 Hopfield 类型。

## 1. 环境与冻结源码

Notebook 固定到一个已经测试过的源码提交，并核对文件哈希。Colab 通常已经包含 JAX；缺少依赖时才安装。

In [ ]:
import hashlib
import importlib.util
from pathlib import Path
import subprocess
import sys
import urllib.request

required = {
    "jax": "jax[cpu]",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

CODE_REV = "4ff9ea8"
CORE_SHA256 = "c40bf773180ccd7ebced4c9413851878ae868e27b0c7207b3d1ff1abd0f79ec2"
CORE_RELATIVE = Path("representation-geometry/experiments/hopfield-dynamic-geometry/experiment_01_boundary_localization.py")

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

candidates = [
    Path.cwd() / CORE_RELATIVE,
    Path.cwd() / "experiment_01_boundary_localization.py",
    Path("/content/nn-labs") / CORE_RELATIVE,
]
core_path = next((path for path in candidates if path.is_file() and sha256(path) == CORE_SHA256), None)
if core_path is None:
    core_path = Path("/content/experiment_01_boundary_localization.py")
    url = (
        "https://raw.githubusercontent.com/Heptazero/nn-labs/"
        f"{CODE_REV}/{CORE_RELATIVE.as_posix()}"
    )
    urllib.request.urlretrieve(url, core_path)

if sha256(core_path) != CORE_SHA256:
    raise RuntimeError("核心源码哈希不匹配；停止运行，避免混用版本")

sys.path.insert(0, str(core_path.parent))
print("core:", core_path)
print("revision:", CODE_REV)
print("sha256:", sha256(core_path))

## 2. 冻结实验条件

- $N=60$；8 个 seed；overlap 为 $-0.5,0,0.5,0.7$；
- $G$ 观察到 $T=6$；独立边界的终点检查观察到 $T=96$；
- 只有终点速度足够小、A/B 终点仍明显分离时，边界才有效；
- 从 $t=2$ 起，要求至少 90% 的切片可识别峰，中位误差不超过 0.01，90% 分位误差不超过 0.025。

预检查没有使用 overlap=0.8：此时两个端点最终合并到同一吸引子，不存在可供定位的两吸引域边界。

In [ ]:
from experiment_01_boundary_localization import (
    BoundaryExperimentConfig,
    run_experiment,
    write_artifacts,
)

config = BoundaryExperimentConfig()
config

## 3. 运行 32 个条件

第一次运行会触发 JAX 编译。CPU 通常只需十几秒。

In [ ]:
from IPython.display import display
import pandas as pd

conditions, time_results, grids = run_experiment(config)
output_dir = (
    Path("/content/experiment_01_outputs")
    if Path("/content").exists()
    else Path("/tmp/nn_labs_experiment_01_outputs")
)
summary, conclusion_path = write_artifacts(
    output_dir, conditions, time_results, grids, config
)

print("output:", output_dir)
display(pd.DataFrame([summary]))
display(pd.read_csv(output_dir / "overlap_summary.csv"))

## 4. 主图

A 显示一个代表条件的完整度量场。B、C 直接检查峰位置和误差；它们重合成水平线不是绘图失败，而是这个等权双记忆系统的交换对称性。D 显示相关性升高时，边界峰的对比度大幅下降。

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(output_dir / "main_figure.png")))

## 5. 怎样判读结果

若 `passed=True`，本实验只允许说：

> 在已经确认存在两个吸引子的对称双记忆系统中，时间切片拉回度量的峰能稳定落在独立终点判据给出的边界上。

但这里的“定位准确”并不难：等权 A/B 交换把边界理论上固定在 $\kappa=0.5$。固定 overlap 后，不同 seed 主要只改变坐标置换和符号，因此不能把 8 个 seed 当作 8 种一般高维几何。相关性扫描提供的有效信息是：overlap 越高，峰越晚达到可识别阈值，而且峰对背景的对比度迅速减弱。

所以通过实验 1 只表示代码和度量定位逻辑完成校准。真正未知形状的边界要等实验 2 的二维输入流形。

## 6. 下载原始产物

输出包括冻结配置、每个条件的边界与终点检查、逐时刻峰值表、完整 $G$ 网格、汇总表、主图和结论草稿。

In [ ]:
import shutil

archive = shutil.make_archive(str(output_dir), "zip", root_dir=output_dir)
print("archive:", archive)

try:
    from google.colab import files
    print("在 Colab 中运行 files.download(archive) 即可下载。")
except ImportError:
    pass

## 方法来源

拉回度量的时间切片构造来自 Pellegrino 与 Chadwick，*RNNs perform task computations by dynamically warping neural representations*（NeurIPS 2025）。这里把同一数学工具用于 Hopfield 检索，但没有复现该论文的 RNN 任务。实验 1 的边界二分、收敛检查和定量门槛是本项目自己的校准协议。

- Paper: https://papers.neurips.cc/paper_files/paper/2025/file/0572c069404875ccca76e822aaf48d50-Paper-Conference.pdf
- Source revision: https://github.com/Heptazero/nn-labs/commit/4ff9ea8